In [0]:
# This Cell is sued to get Secrets we have in Key Vaultss

client_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-client-id-app-reg")
client_secret = dbutils.secrets.get(scope="kv-scope", key="db-secret-value-appregi")
tenant_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-tenant")

# Storage account name
storage_account = "stdehealthcareanalytics"

# OAuth configs
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
# SQL Server connection

sql_server = "healthcare-project-server-2026.database.windows.net"
sql_user = "username"
sql_pass = dbutils.secrets.get(scope="kv-scope", key="sql-pwd-azureportal")
sql_db = "healthcarebootcamp"

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_db}"

connection_properties = {
    "user": sql_user,
    "password": sql_pass,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import col

# Read FilesTable
files_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.FilesTable",
    properties=connection_properties
)

# Filter procedure files
pending_files_df = files_df.filter(
    (col("Status") == "Bronze_Processed") &
    (col("FileName").like("procedures%"))
)

# Create batch list
batch_list = [row.FileName for row in pending_files_df.select("FileName").collect()]

print("Files to process:", batch_list)

# Read from Bronze
bronze_path = "abfss://bronze@stdehealthcareanalytics.dfs.core.windows.net/procedures"

df = spark.read.format("parquet").load(bronze_path)

display(df.limit(10))

Files to process: []


procedure_id,patient_id,visit_id,procedure_name,procedure_date,procedure_cost,procedure_outcome
PRO-17-0000001,PAT-15-0001521,VIS-15-0017193,Appendectomy,2017-06-03,4666.15,Inconclusive
PRO-17-0000002,PAT-15-0002929,VIS-15-0096797,Blood Panel,2018-12-05,5088.65,Follow-up Required
PRO-17-0000003,PAT-15-0001549,NULL,Physiotherapy,2018-11-07,716.39,Inconclusive
PRO-17-0000004,PAT-15-0003711,VIS-15-0049936,Blood Panel,null,3455.11,NULL
null,PAT-15-0001031,NULL,Blood Panel,2019-07-28,5715.71,Follow-up Required
PRO-17-0000006,PAT-15-0003727,VIS-15-0090619,Physiotherapy,2017-12-19,7696.87,Inconclusive
PRO-17-0000007,PAT-15-0004080,VIS-15-0057737,X-Ray Chest,2018-12-04,NULL,Follow-up Required
PRO-17-0000008,PAT-15-0000080,VIS-15-0082868,NaN,2018-05-23,327.79,NULL
PRO-17-0000009,PAT-15-0002069,VIS-15-0025528,NaN,2018-08-22,NULL,NULL
Â PRO-17-0000010,Â PAT-15-0000666,VIS-15-0018105,Appendectomy,2019-08-05,3948.59,Follow-up Required


In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace

# Trim all string columns
for column in df.columns:
    df = df.withColumn(column, trim(col(column)))

# Empty strings to NULL
for column in df.columns:
    df = df.withColumn(column, when(col(column) == "", None).otherwise(col(column)))

# Remove unwanted characters
for column in df.columns:
    df = df.withColumn(
        column,
        when(col(column).isNotNull(),
             regexp_replace(col(column).cast("string"), "Â|\t", "")
        ).otherwise(col(column))
    )

# Replace "NULL" with actual NULL
df = df.replace("NULL", None)

display(df.limit(10))

procedure_id,patient_id,visit_id,procedure_name,procedure_date,procedure_cost,procedure_outcome
PRO-17-0000001,PAT-15-0001521,VIS-15-0017193,Appendectomy,2017-06-03,4666.15,Inconclusive
PRO-17-0000002,PAT-15-0002929,VIS-15-0096797,Blood Panel,2018-12-05,5088.65,Follow-up Required
PRO-17-0000003,PAT-15-0001549,null,Physiotherapy,2018-11-07,716.39,Inconclusive
PRO-17-0000004,PAT-15-0003711,VIS-15-0049936,Blood Panel,null,3455.11,null
null,PAT-15-0001031,null,Blood Panel,2019-07-28,5715.71,Follow-up Required
PRO-17-0000006,PAT-15-0003727,VIS-15-0090619,Physiotherapy,2017-12-19,7696.87,Inconclusive
PRO-17-0000007,PAT-15-0004080,VIS-15-0057737,X-Ray Chest,2018-12-04,null,Follow-up Required
PRO-17-0000008,PAT-15-0000080,VIS-15-0082868,NaN,2018-05-23,327.79,null
PRO-17-0000009,PAT-15-0002069,VIS-15-0025528,NaN,2018-08-22,null,null
PRO-17-0000010,PAT-15-0000666,VIS-15-0018105,Appendectomy,2019-08-05,3948.59,Follow-up Required


In [0]:
from pyspark.sql.functions import col

print("Writing to Silver...")

# Remove NULL primary key
df_valid = df.filter(col("procedure_id").isNotNull())

# Deduplicate
df_valid = df_valid.dropDuplicates(["procedure_id"])

# Silver path
silver_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/procedures/silver"

# Write to Silver
df_valid.write.format("delta").mode("append").save(f"{silver_path}/procedures_silver")

print("Write complete")

Writing to Silver...
Write complete


In [0]:
silver_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/procedures/silver"
spark.read.format("delta").load(f"{silver_path}/procedures_silver").display()


procedure_id,patient_id,visit_id,procedure_name,procedure_date,procedure_cost,procedure_outcome
PRO-17-0000178,PAT-15-0004764,VIS-15-0061788,X-Ray Chest,2019-01-10,736.14,Inconclusive
PRO-17-0000269,PAT-15-0002943,VIS-15-0079984,X-Ray Chest,2018-08-16,7265.3,null
PRO-17-0000481,PAT-15-0004850,VIS-15-0009387,Blood Panel,2017-11-10,371.48,null
PRO-17-0000579,PAT-15-0002209,VIS-15-0094826,X-Ray Chest,2019-01-11,3159.92,Inconclusive
PRO-17-0001015,PAT-15-0001753,VIS-15-0092959,X-Ray Chest,2019-04-18,1179.21,Success
PRO-17-0001339,null,VIS-15-0082899,Appendectomy,null,3150.95,Success
PRO-17-0001476,PAT-15-0002390,VIS-15-0027128,Physiotherapy,2018-09-25,6580.95,Success
PRO-17-0001959,null,VIS-15-0080950,Appendectomy,2017-12-14,6209.07,Success
PRO-17-0002506,PAT-15-0002810,VIS-15-0032564,MRI Scan,2019-06-10,2381.82,Follow-up Required
PRO-17-0002937,PAT-15-0003095,null,MRI Scan,2019-02-02,null,Success


In [0]:
from pyspark.sql.functions import col

# Identify bad records
bad_df = df.filter(
    col("procedure_id").isNull() |
    col("procedure_name").isNull()
)

# Path
bad_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/procedures/badrecords"

# Write bad records
bad_df.write.format("delta").mode("append").save(bad_path)

display(bad_df)

procedure_id,patient_id,visit_id,procedure_name,procedure_date,procedure_cost,procedure_outcome
null,PAT-15-0000976,VIS-15-0025725,NaN,2015-07-09,6438.27,Success
null,PAT-15-0003016,VIS-15-0003482,Physiotherapy,2017-08-15,7806.42,Follow-up Required
null,PAT-15-0001259,VIS-15-0073510,MRI Scan,2017-05-14,4671.64,Inconclusive
null,PAT-15-0003819,VIS-15-0065116,Blood Panel,null,null,Success
null,PAT-15-0002684,VIS-15-0057813,MRI Scan,2017-09-21,3318.65,Success
null,PAT-15-0003683,VIS-15-0053339,NaN,2016-02-11,7882.71,null
null,PAT-15-0004767,VIS-15-0051150,NaN,2016-11-12,7935.39,Follow-up Required
null,PAT-15-0003596,null,NaN,2015-01-23,5426.5,Inconclusive
null,null,null,X-Ray Chest,2016-08-23,7005.29,Follow-up Required
null,PAT-15-0001735,VIS-15-0024059,NaN,2017-04-02,2958.51,Follow-up Required
